<a href="https://colab.research.google.com/github/Theresa-Gao-0529/MSSP607/blob/main/PARTICIPATION_ACTIVITY_Week_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8 participation activity




**Note to the Instructor**

My GitHub ID is **Theresa-Gao-0529**. I previously sent you a collaborator invitation on GitHub, but the invitation has expired. I would greatly appreciate it if you could accept the invitation when you see this note. Thank you for your time and help.

Sincerely,


Xintong Gao

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

preferred_file = Path('InternetSales (2)(1).csv')
if preferred_file.exists():
    data_file = preferred_file
else:
    csv_files = sorted(Path.cwd().glob('InternetSales*.csv'))
    if not csv_files:
        raise FileNotFoundError('Upload InternetSales (2)(1).csv to the notebook folder and run this cell again.')
    data_file = csv_files[0]

sales = pd.read_csv(data_file, encoding='cp1252', dtype={'Suffix': 'string'})
sales['OrderDate'] = pd.to_datetime(sales['OrderDate'], errors='coerce')
sales['SalesAmount'] = pd.to_numeric(sales['SalesAmount'], errors='coerce')
sales = sales.dropna(subset=['OrderDate', 'SalesAmount', 'EnglishProductName']).copy()
sales['Year'] = sales['OrderDate'].dt.year

print(f'Source file: {data_file.name}')
print(f'Rows used: {len(sales):,}')
print(f"Order-date range: {sales['OrderDate'].min():%Y-%m-%d} to {sales['OrderDate'].max():%Y-%m-%d}")

## Annual sales and data coverage


In [ ]:
annual_sales = sales.groupby('Year').agg(
    Annual_Sales=('SalesAmount', 'sum'),
    Months_Observed=('OrderDate', lambda dates: dates.dt.to_period('M').nunique()),
    Order_Lines=('SalesAmount', 'size')
).sort_index()
annual_sales['Coverage'] = np.where(annual_sales['Months_Observed'].eq(12), 'Complete (12 months)', 'Partial year')
display(annual_sales.style.format({'Annual_Sales': '${:,.2f}', 'Months_Observed': '{:.0f}', 'Order_Lines': '{:,.0f}'}))

complete_years = annual_sales.loc[annual_sales['Months_Observed'].eq(12), 'Annual_Sales']
if len(complete_years) < 2:
    raise ValueError('At least two complete calendar years are required to fit a linear trend.')
print(f'Complete years used for the trend: {complete_years.index.min()}–{complete_years.index.max()}')

## Product sales summary



In [ ]:
product_summary = (
    sales.groupby('EnglishProductName', as_index=False)
    .agg(Total_Sales=('SalesAmount', 'sum'), Order_Lines=('SalesAmount', 'size'),
         Mean_Line_Sales=('SalesAmount', 'mean'))
    .sort_values('Total_Sales', ascending=False)
    .reset_index(drop=True)
)
top_product = product_summary.iloc[0]
print(f"Highest total sales: {top_product['EnglishProductName']} — ${top_product['Total_Sales']:,.2f}")
display(product_summary.head(10).style.format({'Total_Sales': '${:,.2f}', 'Mean_Line_Sales': '${:,.2f}', 'Order_Lines': '{:,.0f}'}))

## Five-year linear trend forecast



In [ ]:
year_origin = int(complete_years.index.min())
x = (complete_years.index - year_origin).to_numpy(dtype=float)
y = complete_years.to_numpy(dtype=float)
slope, intercept = np.polyfit(x, y, 1)
last_observed_year = int(sales['OrderDate'].max().year)
forecast_years = np.arange(last_observed_year + 1, last_observed_year + 6)
forecast_values = slope * (forecast_years - year_origin) + intercept
forecast = pd.DataFrame({'Year': forecast_years, 'Forecast_Sales': forecast_values})
forecast['Forecast_Sales'] = forecast['Forecast_Sales'].clip(lower=0)

print(f'Linear trend equation: annual sales = {slope:,.2f} × (year − {year_origin}) + {intercept:,.2f}')
display(forecast.style.format({'Forecast_Sales': '${:,.2f}'}))

trend_years = np.arange(year_origin, int(forecast_years.max()) + 1)
trend_values = slope * (trend_years - year_origin) + intercept
plt.figure(figsize=(10, 6))
plt.scatter(complete_years.index.to_numpy(dtype=float), y, color='#1f77b4', s=65, label='Actual annual sales (complete years)', zorder=3)
plt.plot(trend_years, trend_values, color='#d62728', linewidth=2.5, label='Linear trend and forecast')
plt.axvline(last_observed_year + 0.5, color='gray', linestyle='--', linewidth=1, label='Forecast begins')
plt.title('Internet Sales: Annual Sales and Five-Year Linear Forecast')
plt.xlabel('Calendar year')
plt.ylabel('Sales amount')
plt.xticks(trend_years)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f'${value/1_000_000:.0f}M'))
plt.grid(axis='y', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
first_forecast = forecast.iloc[0]
last_forecast = forecast.iloc[-1]
print(f"The highest-sales product is {top_product['EnglishProductName']}, with historical sales of ${top_product['Total_Sales']:,.2f}.")
print(f"The linear trend forecasts ${first_forecast['Forecast_Sales']:,.2f} for {int(first_forecast['Year'])} and ${last_forecast['Forecast_Sales']:,.2f} for {int(last_forecast['Year'])}.")
print('These are illustrative trend extrapolations from complete calendar years, not causal or seasonally adjusted predictions.')